# 1. ORM 구조

In [208]:
from sqlalchemy import create_engine, Column, Integer, String, Float, ForeignKey
from sqlalchemy.orm import relationship, sessionmaker, declarative_base

## 1. 엔진 및 세션 설정

In [209]:
# 1. 엔진 및 세션 설정
# SQLite를 사용하며, 파일명은 crime_data.db로 설정합니다.
engine = create_engine('sqlite:///crime_data.db', echo=False)
Session = sessionmaker(bind=engine)
session = Session()
Base = declarative_base()

## 2. 테이블 생성

In [210]:
# --- [ 마스터 테이블 ] ---

class RegionMaster(Base):
    """지역 마스터 (예: 서울 강남구, 서울 종로구)"""
    __tablename__ = 'region_master'
    id = Column(Integer, primary_key=True, autoincrement=True)
    region_name = Column(String, unique=True, nullable=False)

    # 관계 설정
    mappers = relationship("RegionMapper", back_populates="region")
    region_crimes = relationship("CrimeRegion", back_populates="region")

class CrimeCategory(Base):
    """범죄 종류 마스터 (예: 절도, 폭행)"""
    __tablename__ = 'crime_category'
    id = Column(Integer, primary_key=True, autoincrement=True)
    main_cat = Column(String)  # 범죄대분류
    sub_cat = Column(String, unique=True) # 범죄중분류

    # 관계 설정
    region_stats = relationship("CrimeRegion", back_populates="category")
    time_stats = relationship("CrimeTime", back_populates="category")
    week_stats = relationship("CrimeWeek", back_populates="category")

# --- [ 매핑 테이블 (핵심 연결 고리) ] ---

class RegionMapper(Base):
    """핫스팟과 지역 마스터를 잇는 중간 다리 (mapping_fix 기반)"""
    __tablename__ = 'region_mapper'
    id = Column(Integer, primary_key=True, autoincrement=True)
    AREA_GU = Column(String)
    CATEGORY = Column(String)
    NO = Column(Integer)
    AREA_CD = Column(String)
    AREA_NM = Column(String)
    ENG_NM = Column(String)
    
    # FK 설정
    hotspot_id = Column(Integer, ForeignKey('hotspot_api.id'))
    region_id = Column(Integer, ForeignKey('region_master.id'))

    # 관계 설정
    hotspot = relationship("HotspotAPI", back_populates="mapper")
    region = relationship("RegionMaster", back_populates="mappers")

# --- [ 데이터 테이블 ] ---

class HotspotAPI(Base):
    """핫스팟 API 정보 (seoul live cache 기반)"""
    __tablename__ = 'hotspot_api'
    id = Column(Integer, primary_key=True, autoincrement=True)
    area_name = Column(String)
    congest_lvl = Column(Integer)
    ppltn_min = Column(Integer)
    ppltn_max = Column(Integer)
    temp = Column(Float)
    update_time = Column(String)
    collected_at = Column(String)

    # 매핑 테이블과 1:1 연결
    mapper = relationship("RegionMapper", back_populates="hotspot", uselist=False)

class CrimeRegion(Base):
    """경찰청 지역별 범죄 통계 (police_region_fix 기반)"""
    __tablename__ = 'crime_region'
    id = Column(Integer, primary_key=True, autoincrement=True)
    crime_count = Column(Integer)
    
    region_id = Column(Integer, ForeignKey('region_master.id'))
    category_id = Column(Integer, ForeignKey('crime_category.id'))
    
    region = relationship("RegionMaster", back_populates="region_crimes")
    category = relationship("CrimeCategory", back_populates="region_stats")

class CrimeTime(Base):
    """시간대별 범죄 통계 (전국 통계용)"""
    __tablename__ = 'crime_time'
    id = Column(Integer, primary_key=True, autoincrement=True)
    time_range = Column(String)
    crime_count = Column(Float)
    
    category_id = Column(Integer, ForeignKey('crime_category.id'))
    category = relationship("CrimeCategory", back_populates="time_stats")

class CrimeWeek(Base):
    """요일별 범죄 통계 (전국 통계용)"""
    __tablename__ = 'crime_week'
    id = Column(Integer, primary_key=True, autoincrement=True)
    day_of_week = Column(String)
    crime_count = Column(Integer)
    
    category_id = Column(Integer, ForeignKey('crime_category.id'))
    category = relationship("CrimeCategory", back_populates="week_stats")

In [211]:
# 2. 테이블 생성
Base.metadata.drop_all(engine)
Base.metadata.create_all(engine)
print("의도하신 매핑 구조로 DB 생성 완료!")

의도하신 매핑 구조로 DB 생성 완료!


# 2. 데이터 삽입

## 리전 마스터


In [212]:
import pandas as pd

df_region_master = pd.read_csv('../data/police_region_fix.csv', encoding='utf-8-sig')


In [213]:
df_region_master.head()

,범죄대분류,범죄중분류,지역,범죄건수
0,강력범죄,살인기수,서울 종로구,0
1,강력범죄,살인미수등,서울 종로구,1
2,강력범죄,강도,서울 종로구,5
3,강력범죄,강간,서울 종로구,31
4,절도범죄,절도,서울 종로구,1180


In [214]:
insert_master = list(df_region_master['지역'].unique())

In [215]:
for region in insert_master:
    new_region = RegionMaster(region_name=region)
    session.add(new_region)
session.commit()

## 크라임 카테고리

In [216]:
df_crime_category = pd.read_csv('../data/police_region_fix.csv', encoding='utf-8-sig')

In [217]:
insert_category = list(df_crime_category[['범죄대분류', '범죄중분류']].drop_duplicates().itertuples(index=False, name=None))

insert_category 

[('강력범죄', '살인기수'),
 ('강력범죄', '살인미수등'),
 ('강력범죄', '강도'),
 ('강력범죄', '강간'),
 ('절도범죄', '절도'),
 ('폭력범죄', '상해'),
 ('폭력범죄', '폭행'),
 ('폭력범죄', '협박'),
 ('마약범죄', '마약')]

In [218]:
for crime in insert_category:
    new_category = CrimeCategory(main_cat=crime[0], sub_cat=crime[1])
    session.add(new_category)
session.commit()

## API 테이블 (서울라이브)

In [219]:
df_seoul_live = pd.read_csv('../data/seoul_live_cache.csv', encoding='utf-8-sig')
df_seoul_live.head()

,area_name,congest_lvl,ppltn_min,ppltn_max,temp,update_time,collected_at
0,강남 MICE 관광특구,약간 붐빔,24000,26000,8.7,2026-03-09 14:35,2026-03-09 15:00:32
1,동대문 관광특구,약간 붐빔,32000,34000,7.2,2026-03-09 14:35,2026-03-09 15:00:32
2,명동 관광특구,약간 붐빔,115000,120000,7.2,2026-03-09 14:35,2026-03-09 15:00:33
3,이태원 관광특구,약간 붐빔,10000,12000,7.0,2026-03-09 14:35,2026-03-09 15:00:33
4,잠실 관광특구,여유,52000,54000,8.2,2026-03-09 14:35,2026-03-09 15:00:33


In [220]:
insert_seoul_live = list(df_seoul_live[['area_name', 'congest_lvl', 'ppltn_min', 'ppltn_max', 'temp', 'update_time', 'collected_at']].itertuples(index=False, name=None))
insert_seoul_live

[('강남 MICE 관광특구',
  '약간 붐빔',
  24000,
  26000,
  8.7,
  '2026-03-09 14:35',
  '2026-03-09 15:00:32'),
 ('동대문 관광특구',
  '약간 붐빔',
  32000,
  34000,
  7.2,
  '2026-03-09 14:35',
  '2026-03-09 15:00:32'),
 ('명동 관광특구',
  '약간 붐빔',
  115000,
  120000,
  7.2,
  '2026-03-09 14:35',
  '2026-03-09 15:00:33'),
 ('이태원 관광특구',
  '약간 붐빔',
  10000,
  12000,
  7.0,
  '2026-03-09 14:35',
  '2026-03-09 15:00:33'),
 ('잠실 관광특구',
  '여유',
  52000,
  54000,
  8.2,
  '2026-03-09 14:35',
  '2026-03-09 15:00:33'),
 ('종로·청계 관광특구',
  '약간 붐빔',
  50000,
  52000,
  7.2,
  '2026-03-09 14:35',
  '2026-03-09 15:00:34'),
 ('홍대 관광특구',
  '여유',
  74000,
  76000,
  6.9,
  '2026-03-09 14:35',
  '2026-03-09 15:00:34'),
 ('경복궁', '여유', 2000, 2500, 7.2, '2026-03-09 14:35', '2026-03-09 15:00:34'),
 ('광화문·덕수궁',
  '약간 붐빔',
  58000,
  60000,
  7.2,
  '2026-03-09 14:35',
  '2026-03-09 15:00:35'),
 ('보신각',
  '약간 붐빔',
  24000,
  26000,
  7.2,
  '2026-03-09 14:35',
  '2026-03-09 15:00:35'),
 ('서울 암사동 유적', '여유', 300, 400, 6.7, '2026-03-09 1

In [221]:
for seoul in insert_seoul_live:
    new_seoul = HotspotAPI(area_name=seoul[0], congest_lvl=seoul[1], ppltn_min=seoul[2], ppltn_max=seoul[3], temp=seoul[4], update_time=seoul[5], collected_at=seoul[6])
    session.add(new_seoul)
session.commit()


##  매핑 테이블

In [222]:
def fill_region_mapper():
    print("RegionMapper ID 매핑 시작...")
    
    # 1. DB에서 마스터 ID 정보 로드
    # (문자열 이름을 키로 숫자인 ID를 가져옵니다)
    hotspot_db = pd.read_sql("SELECT id as hotspot_id, area_name FROM hotspot_api", engine)
    region_db = pd.read_sql("SELECT id as region_id, region_name FROM region_master", engine)
    
    # 2. 기준 데이터(CSV) 로드
    df_csv = pd.read_csv('../data/mapping_fix.csv', encoding='utf-8-sig')

    # 3. 데이터 결합 (Merge)
    # 핫스팟 이름(AREA_NM)으로 hotspot_id 가져오기
    df_merged = pd.merge(df_csv, hotspot_db, left_on='AREA_NM', right_on='area_name', how='left')
    
    # 지역 이름(AREA_GU)으로 region_id 가져오기
    df_final = pd.merge(df_merged, region_db, left_on='AREA_GU', right_on='region_name', how='left')

    # 4. DB 컬럼명에 맞게 정리
    # 모델 정의와 CSV 컬럼명을 일치시킵니다.
    final_mapper = df_final[[
        'AREA_GU', 'CATEGORY', 'NO', 'AREA_CD', 'AREA_NM', 'ENG_NM', 
        'hotspot_id', 'region_id'
    ]]

    # 5. DB 업데이트 (if_exists='replace'를 써서 기존 NULL 데이터를 덮어씌웁니다)
    # 주의: replace를 쓰면 테이블이 삭제되었다가 다시 생성되므로, 
    # 이후에는 append로 바꾸거나 6단계 코드를 참고하세요.
    final_mapper.to_sql('region_mapper', con=engine, if_exists='replace', index=False)
    
    print(f"매핑 테이블 업데이트 완료! ({len(final_mapper)}건)")

# 실행
fill_region_mapper()

RegionMapper ID 매핑 시작...
매핑 테이블 업데이트 완료! (123건)


## 경찰청 지역

- 범죄건수만 넣으면 됨 나머지는 끌어와야함

In [223]:
df_crime_category

,범죄대분류,범죄중분류,지역,범죄건수
0,강력범죄,살인기수,서울 종로구,0
1,강력범죄,살인미수등,서울 종로구,1
2,강력범죄,강도,서울 종로구,5
3,강력범죄,강간,서울 종로구,31
4,절도범죄,절도,서울 종로구,1180
...,...,...,...,...
229,절도범죄,절도,경기도 과천시,327
230,폭력범죄,상해,경기도 과천시,11
231,폭력범죄,폭행,경기도 과천시,106
232,폭력범죄,협박,경기도 과천시,24


In [224]:
def insert_crime_region_stats():
    # 1. 마스터 ID 맵들 가져오기
    region_map = pd.read_sql("SELECT id as region_id, region_name FROM region_master", engine)
    category_map = pd.read_sql("SELECT id as category_id, sub_cat FROM crime_category", engine)

    # 2. 지역별 범죄 데이터 로드 (police_region_fix.csv)
    df_crime = pd.read_csv('../data/police_region_fix.csv')

    # 3. 지역명으로 region_id 합치기
    # csv의 '지역' 컬럼과 마스터의 'region_name'을 매칭
    df_mapped = pd.merge(df_crime, region_map, left_on='지역', right_on='region_name')

    # 4. 범죄중분류명으로 category_id 합치기
    # csv의 '범죄중분류' 컬럼과 마스터의 'sub_cat'을 매칭
    df_final = pd.merge(df_mapped, category_map, left_on='범죄중분류', right_on='sub_cat')

    # 5. 필요한 컬럼만 추출해서 DB로 쏘기
    crime_region_to_db = df_final[['범죄건수', 'region_id', 'category_id']]
    crime_region_to_db.columns = ['crime_count', 'region_id', 'category_id']
    
    crime_region_to_db.to_sql('crime_region', con=engine, if_exists='append', index=False)
    print(f"CrimeRegion 통계 {len(crime_region_to_db)}건 삽입 완료!")

insert_crime_region_stats()

CrimeRegion 통계 234건 삽입 완료!


## 경찰청 시간

In [225]:
def insert_crime_time_stats():
    print("CrimeTime 데이터 삽입 시작...")
    # 1. 범죄 카테고리 ID 맵 가져오기
    category_map = pd.read_sql("SELECT id as category_id, sub_cat FROM crime_category", engine)

    # 2. 시간대별 범죄 데이터 로드
    df_time = pd.read_csv('../data/police_time_fix.csv')

    # 3. 범죄중분류명으로 category_id 합치기 (Join)
    df_time_mapped = pd.merge(df_time, category_map, left_on='범죄중분류', right_on='sub_cat')

    # 4. 테이블 구조에 맞게 정리
    # 컬럼 매핑: 시간대2 -> time_range, 범죄건수 -> crime_count
    time_to_db = df_time_mapped[['시간대2', '범죄건수', 'category_id']]
    time_to_db.columns = ['time_range', 'crime_count', 'category_id']

    # 5. DB 삽입
    time_to_db.to_sql('crime_time', con=engine, if_exists='append', index=False)
    print(f"CrimeTime 완료: {len(time_to_db)}건 삽입됨.")

insert_crime_time_stats()

CrimeTime 데이터 삽입 시작...
CrimeTime 완료: 72건 삽입됨.


## 경찰청 요일

In [226]:
def insert_crime_week_stats():
    print("CrimeWeek 데이터 삽입 시작...")
    # 1. 범죄 카테고리 ID 맵 가져오기
    category_map = pd.read_sql("SELECT id as category_id, sub_cat FROM crime_category", engine)

    # 2. 요일별 범죄 데이터 로드
    df_week = pd.read_csv('../data/police_week_fix.csv')

    # 3. 범죄중분류명으로 category_id 합치기 (Join)
    df_week_mapped = pd.merge(df_week, category_map, left_on='범죄중분류', right_on='sub_cat')

    # 4. 테이블 구조에 맞게 정리
    # 컬럼 매핑: 요일 -> day_of_week, 발생건수 -> crime_count
    week_to_db = df_week_mapped[['요일', '발생건수', 'category_id']]
    week_to_db.columns = ['day_of_week', 'crime_count', 'category_id']

    # 5. DB 삽입
    week_to_db.to_sql('crime_week', con=engine, if_exists='append', index=False)
    print(f"CrimeWeek 완료: {len(week_to_db)}건 삽입됨.")

insert_crime_week_stats()

CrimeWeek 데이터 삽입 시작...
CrimeWeek 완료: 63건 삽입됨.
